# Retail Sales Forecasting

This notebook presents a complete data science understanding. The project focuses on forecasting weekly sales for a retail chain using historical sales data, store characteristics, and external factors.

## 1. Problem Definition & Business Understanding

- **Objective:** Forecast weekly sales and understand key drivers (e.g., markdown promotions, holidays, economic factors) affecting sales performance.
- **Key Questions:**
  - What factors (store type, markdowns, holidays, economic indicators) drive sales?
  - How do promotional markdowns impact weekly sales?
  - Can we predict future sales accurately using historical data?
- **Success Metrics:** RMSE, MAE, R², and actionable insights for decision making.

### Import necessary Libraries

In [396]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
 
from sklearn.impute import SimpleImputer 
from sklearn.preprocessing import LabelEncoder 
 
import scipy.stats as stats 

from sklearn.preprocessing import LabelEncoder, StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import warnings 
warnings.filterwarnings('ignore') 

## 2. Data Integration

We have three datasets:

1. **features.csv**: Contains environmental and economic data (Temperature, Fuel_Price, CPI, Unemployment, MarkDowns, and IsHoliday).
2. **train.csv**: Contains weekly sales data (Store, Dept, Date, Weekly_Sales, IsHoliday).
3. **stores.csv**: Contains store characteristics (Store, Type, Size).

The datasets are merged using common keys (Store) to produce a final dataset for analysis. In this notebook, we assume the merged dataset is available as `df`.

In [397]:
features = pd.read_csv("features.csv.xls")

In [398]:
features.shape

(8190, 12)

**In Features has 8190 rows and 12 columns**

In [399]:
stores = pd.read_csv("stores.csv.xls")

In [400]:
stores.shape

(45, 3)

**In Store has 45 rows and 3 columns**

In [401]:
train = pd.read_csv("train.csv.xls")

In [402]:
train.shape

(421570, 5)

**In train has 421570 rows and 5 columns**

In [403]:
# Merge train with features on 'Store'
df = pd.merge(train, features, how="inner")

In [404]:
# Merge the result with stores on 'Store'
df = pd.merge(df, stores, on="Store", how="inner")

In [405]:
df.shape

(421570, 16)

**df DataFrame has 421570 and 16 rows in initial phase**

In [406]:
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [407]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 16 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
 5   Temperature   421570 non-null  float64
 6   Fuel_Price    421570 non-null  float64
 7   MarkDown1     150681 non-null  float64
 8   MarkDown2     111248 non-null  float64
 9   MarkDown3     137091 non-null  float64
 10  MarkDown4     134967 non-null  float64
 11  MarkDown5     151432 non-null  float64
 12  CPI           421570 non-null  float64
 13  Unemployment  421570 non-null  float64
 14  Type          421570 non-null  object 
 15  Size          421570 non-null  int64  
dtypes: bool(1), float64(10), int64(3), object(2)
memory usage: 48.6+ MB


**<pre>Bool Datatype: IsHoliday
Object Dataype: Type
Date type: Date 
Nulls: All MarkDown</pre>**

## 3. Data Cleaning & Preprocessing

- **Convert data types** (e.g., ensure the Date column is datetime).
- **Check for missing values** and handle them appropriately (e.g., fill missing markdowns with 0 if no discount was applied).

In [408]:
# Convert the Date column to datetime format
df["Date"] = pd.to_datetime(df["Date"])
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


In [409]:
df.isnull().sum()

Store                0
Dept                 0
Date                 0
Weekly_Sales         0
IsHoliday            0
Temperature          0
Fuel_Price           0
MarkDown1       270889
MarkDown2       310322
MarkDown3       284479
MarkDown4       286603
MarkDown5       270138
CPI                  0
Unemployment         0
Type                 0
Size                 0
dtype: int64

In [410]:
df[['MarkDown1', 'MarkDown2','MarkDown3','MarkDown4','MarkDown5']].isnull().sum().mean()

np.float64(284486.2)

**Around 284486 Weeks are not any MarkDown scheme is apply**

In [411]:
# fill the nulls whit 0
df[['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']] = df[['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']].fillna(0)

In [412]:
df.isnull().sum()

Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
Temperature     0
Fuel_Price      0
MarkDown1       0
MarkDown2       0
MarkDown3       0
MarkDown4       0
MarkDown5       0
CPI             0
Unemployment    0
Type            0
Size            0
dtype: int64

In [413]:
df = df.loc[df['Weekly_Sales'] > 0]

## 4. Exploratory Data Analysis (EDA)
We perform univariate, bivariate, and multivariate analyses along with computing central tendency, dispersion, skewness, kurtosis, covariance, and correlation.

In [414]:
num_columns_means = df.select_dtypes(include=np.number).mean()
num_columns_means

Store               22.195611
Dept                44.241309
Weekly_Sales     16033.114591
Temperature         60.090599
Fuel_Price           3.360890
MarkDown1         2590.323565
MarkDown2          878.905242
MarkDown3          468.845949
MarkDown4         1083.534361
MarkDown5         1662.805002
CPI                171.212496
Unemployment         7.960000
Size            136749.732787
dtype: float64

In [415]:
num_columns_median = df.select_dtypes(include=np.number).median()
num_columns_median

Store               22.000000
Dept                37.000000
Weekly_Sales      7661.700000
Temperature         62.090000
Fuel_Price           3.452000
MarkDown1            0.000000
MarkDown2            0.000000
MarkDown3            0.000000
MarkDown4            0.000000
MarkDown5            0.000000
CPI                182.350989
Unemployment         7.866000
Size            140167.000000
dtype: float64

In [416]:
mode_values = df.mode().iloc[0]
mode_values

Store                          13.0
Dept                              1
Date            2011-12-23 00:00:00
Weekly_Sales                   10.0
IsHoliday                     False
Temperature                   50.43
Fuel_Price                    3.638
MarkDown1                       0.0
MarkDown2                       0.0
MarkDown3                       0.0
MarkDown4                       0.0
MarkDown5                       0.0
CPI                      129.855533
Unemployment                  8.099
Type                              A
Size                        39690.0
Name: 0, dtype: object

In [417]:
df.drop(['Date','Type'], axis=1).skew()

Store            0.077947
Dept             0.359016
Weekly_Sales     3.258942
IsHoliday        3.360255
Temperature     -0.321295
Fuel_Price      -0.104678
MarkDown1        4.730933
MarkDown2       10.649277
MarkDown3       14.908890
MarkDown4        8.075125
MarkDown5        9.952112
CPI              0.084673
Unemployment     1.183789
Size            -0.326689
dtype: float64

In [418]:
df.drop(['Date','Type'], axis=1).kurt()

Store            -1.146698
Dept             -1.216644
Weekly_Sales     21.460793
IsHoliday         9.291355
Temperature      -0.636128
Fuel_Price       -1.185445
MarkDown1        34.912307
MarkDown2       145.568291
MarkDown3       247.647517
MarkDown4        86.188903
MarkDown5       183.080984
CPI              -1.829848
Unemployment      2.729259
Size             -1.206400
dtype: float64

In [419]:
df.drop(['Date','Type'], axis=1).corr()

,Store,Dept,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Size
Store,1.000000,0.024258,-0.085117,-0.000522,-0.050230,0.065321,-0.059970,-0.033727,-0.020301,-0.042808,-0.012605,-0.211261,0.208759,-0.182763
Dept,0.024258,1.000000,0.148749,0.000663,0.004727,0.003544,0.001454,0.000435,0.001517,0.001881,0.002766,-0.007178,0.007787,-0.002491
Weekly_Sales,-0.085117,0.148749,1.000000,0.012843,-0.002339,0.000089,0.047249,0.020948,0.038522,0.037470,0.050598,-0.021162,-0.025806,0.244117
IsHoliday,-0.000522,0.000663,0.012843,1.000000,-0.155775,-0.078155,-0.003517,0.207326,0.266731,0.011625,-0.015186,-0.001933,0.010555,0.000797
Temperature,-0.050230,0.004727,-0.002339,-0.155775,1.000000,0.143700,-0.026428,-0.179684,-0.056016,-0.050323,-0.014847,0.182223,0.096768,-0.058413
Fuel_Price,0.065321,0.003544,0.000089,-0.078155,0.143700,1.000000,0.297075,0.029282,0.018646,0.166645,0.215588,-0.164199,-0.033915,0.003632
MarkDown1,-0.059970,0.001454,0.047249,-0.003517,-0.026428,0.297075,1.000000,0.175005,-0.014425,0.838866,0.415271,0.010915,-0.105257,0.169891
MarkDown2,-0.033727,0.000435,0.020948,0.207326,-0.179684,0.029282,0.175005,1.000000,-0.006090,0.113446,0.131847,-0.003694,-0.041497,0.078392
MarkDown3,-0.020301,0.001517,0.038522,0.266731,-0.056016,0.018646,-0.014425,-0.006090,1.000000,-0.012031,0.042542,-0.005959,-0.018078,0.033671
MarkDown4,-0.042808,0.001881,0.037470,0.011625,-0.050323,0.166645,0.838866,0.113446,-0.012031,1.000000,0.303536,-0.002061,-0.076583,0.127415


In [420]:
numerical_columns = df.select_dtypes(include=np.number).columns

In [421]:
# num_cols = len(numerical_columns)
# plt.figure(figsize=(15, 5 * num_cols))
# for i, col in enumerate(numerical_columns, 1):
#     # if col == "Weekly_Sales":
#     #     continue
#     plt.subplot(num_cols, 3, i)
#     plt.boxplot(df[col])
#     plt.title(f'Boxplot of {col}')
#     plt.ylabel('Values')

# plt.tight_layout()
# plt.show()

In [422]:
num_data = df.select_dtypes(include = np.number)
Q1 = num_data.quantile(0.25)
Q3 = num_data.quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - (1.5 * IQR)
upper_bound = Q3 + (1.5 * IQR)

In [423]:
# df = df[(df['Store'] > lower_bound['Store']) & (df['Store'] < upper_bound['Store'])]
# df.reset_index(drop=True, inplace=True)

In [424]:
# sns.boxplot(df['Store'])
# plt.show

In [425]:
# df = df[(df['Dept'] > lower_bound['Dept']) & (df['Dept'] < upper_bound['Dept'])]
# df.reset_index(drop=True, inplace=True)

In [426]:
# df = df[(df['Weekly_Sales'] > lower_bound['Weekly_Sales']) & (df['Weekly_Sales'] < upper_bound['Weekly_Sales'])]
# df.reset_index(drop=True, inplace=True)

In [427]:
# df = df[(df['Temperature'] > lower_bound['Temperature']) & (df['Temperature'] < upper_bound['Temperature'])]
# df.reset_index(drop=True, inplace=True)

In [428]:
# df = df[(df['Fuel_Price'] > lower_bound['Fuel_Price']) & (df['Fuel_Price'] < upper_bound['Fuel_Price'])]
# df.reset_index(drop=True, inplace=True)

In [429]:
# df = df[(df['MarkDown1'] > lower_bound['MarkDown1']) & (df['MarkDown1'] < upper_bound['MarkDown1'])]
# df.reset_index(drop=True, inplace=True)

In [430]:
# df = df[(df['MarkDown2'] > lower_bound['MarkDown2']) & (df['MarkDown2'] < upper_bound['MarkDown2'])]
# df.reset_index(drop=True, inplace=True)

In [431]:
# df = df[(df['MarkDown3'] > lower_bound['MarkDown3']) & (df['MarkDown3'] < upper_bound['MarkDown3'])]
# df.reset_index(drop=True, inplace=True)

In [432]:
# df = df[(df['MarkDown4'] > lower_bound['MarkDown4']) & (df['MarkDown4'] < upper_bound['MarkDown4'])]
# df.reset_index(drop=True, inplace=True)

In [433]:
# df = df[(df['MarkDown5'] > lower_bound['MarkDown5']) & (df['MarkDown5'] < upper_bound['MarkDown5'])]
# df.reset_index(drop=True, inplace=True)

In [434]:
# df = df[(df['CPI'] > lower_bound['CPI']) & (df['CPI'] < upper_bound['CPI'])]
# df.reset_index(drop=True, inplace=True)

In [435]:
# df = df[(df['Unemployment'] > lower_bound['Unemployment']) & (df['Unemployment'] < upper_bound['Unemployment'])]
# df.reset_index(drop=True, inplace=True)

In [436]:
# df = df[(df['Size'] > lower_bound['Size']) & (df['Size'] < upper_bound['Size'])]
# df.reset_index(drop=True, inplace=True)

In [437]:
df['Weekly_Sales']


0         24924.50
1         46039.49
2         41595.55
3         19403.54
4         21827.90
            ...   
421565      508.37
421566      628.10
421567     1061.02
421568      760.01
421569     1076.80
Name: Weekly_Sales, Length: 420212, dtype: float64

In [439]:
# plt.figure(figsize=[20,10])
# sns.heatmap(df.drop(['Date','Type'], axis=1).corr(), annot = True)

In [440]:
# # Plot histogram & KDE for Weekly Sales
# plt.figure(figsize=(10, 5))
# sns.histplot(df['Weekly_Sales'], kde=True, color='blue')
# plt.title('Distribution of Weekly Sales')
# plt.xlabel('Weekly Sales')
# plt.ylabel('Frequency')
# plt.show()


In [441]:
# plt.figure(figsize=(12, 5))
# sns.histplot(df['CPI'], bins=50, kde=True)
# plt.title('Distribution of CPI')
# plt.xlabel('CPI')
# plt.ylabel('Frequency')
# plt.show()

In [442]:
# plt.figure(figsize=(12, 5))
# sns.histplot(df['Unemployment'], bins=50, kde=True)
# plt.title('Distribution of Unemployment')
# plt.xlabel('Unemployment Rate')
# plt.ylabel('Frequency')
# plt.show()

In [443]:
# # Bivariate Analysis
# plt.figure(figsize=(12, 6))
# sns.scatterplot(x=df['Temperature'], y=df['Weekly_Sales'])
# plt.title('Weekly Sales vs Temperature')
# plt.xlabel('Temperature')
# plt.ylabel('Weekly Sales')
# plt.show()

In [444]:
# plt.figure(figsize=(6, 4))
# sns.barplot(x="IsHoliday", y="Weekly_Sales", data=df)
# plt.title("Holiday vs. Non-Holiday Sales")
# plt.show()

In [445]:
# plt.figure(figsize=(12, 6))
# sns.scatterplot(x=df['Fuel_Price'], y=df['CPI'])
# plt.title('CPI vs Fuel Price')
# plt.xlabel('Fuel Price')
# plt.ylabel('CPI')
# plt.show()

In [446]:
# plt.figure(figsize=(12, 6))
# sns.scatterplot(x=df['Size'], y=df['Weekly_Sales'])
# plt.title('Store Size vs Weekly Sales')
# plt.xlabel('Store Size')
# plt.ylabel('Weekly Sales')
# plt.show()

In [447]:
# plt.figure(figsize=(8, 6))
# sns.boxplot(x='Type', y='Weekly_Sales', data=df)
# plt.title('Weekly Sales by Store Type')
# plt.show()

In [448]:
#Multivariate

In [449]:
# plt.figure(figsize=(12, 6))
# sns.heatmap(df[['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']].corr(), annot=True)
# plt.title('Correlation Heatmap')
# plt.show()

In [450]:
# sns.pairplot(df[['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment']])
# plt.show()

In [451]:
# sns.heatmap(df[['Weekly_Sales', 'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']].corr(), annot = True)

In [452]:
# sns.lineplot(x=df["MarkDown1"], y=df["MarkDown5"])

In [453]:
# plt.plot(df["MarkDown5"], label='Line 1')
# plt.plot(df["MarkDown1"], label='Line 2')
# plt.xlabel('X-axis')
# plt.ylabel('Y-axis')
# plt.legend()
# plt.show()

In [454]:
# sns.lineplot(x=df["MarkDown1"],y=df["Weekly_Sales"], label='Line 1')
# sns.lineplot(x=df["MarkDown5"],y=df["Weekly_Sales"], label='Line 2')
# plt.show()

In [455]:
# sns.scatterplot(x=df["MarkDown1"], y=df["Weekly_Sales"])
# plt.title("Weekly Sales vs. MarkDown1")
# plt.show()

## 5. Feature Engineering

Create additional features such as date-based components.

In [457]:
# Extract date features
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['Week'] = df['Date'].dt.isocalendar().week
df['Day'] = df['Date'].dt.day

In [458]:
df.drop(['Date'], axis=1, inplace = True)

## 5. Encoding, Scaling and VIF

- **Label Encoding:** Used for `IsHoliday` and `Type` since they are categorical.
- Define the target (`Weekly_Sales`) and features
- **Standard Scaling:** Applied `StandardScaler` to normalize numerical features (`Weekly Sales`, `Temperature`, `Fuel Price`, etc.) to improve ML model performance.
- 
**VIF Calculation:**- Measures multicollinearity among numerical variables.
- High VIF values (>10) indicate strong multicollinearity, requiring feature selection.

In [459]:
n_data = df.copy()

In [460]:
n_data.head()

,Store,Dept,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size,Year,Month,Week,Day
0,1,1,24924.50,False,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.096358,8.106,A,151315,2010,2,5,5
1,1,1,46039.49,True,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.242170,8.106,A,151315,2010,2,6,12
2,1,1,41595.55,False,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.289143,8.106,A,151315,2010,2,7,19
3,1,1,19403.54,False,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.319643,8.106,A,151315,2010,2,8,26
4,1,1,21827.90,False,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.350143,8.106,A,151315,2010,3,9,5


In [461]:
# Encoding categorical variables
label_encoder = LabelEncoder()
n_data['IsHoliday'] = label_encoder.fit_transform(n_data['IsHoliday'])
n_data['Type'] = label_encoder.fit_transform(n_data['Type'])

In [462]:
n_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 420212 entries, 0 to 421569
Data columns (total 19 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         420212 non-null  int64  
 1   Dept          420212 non-null  int64  
 2   Weekly_Sales  420212 non-null  float64
 3   IsHoliday     420212 non-null  int64  
 4   Temperature   420212 non-null  float64
 5   Fuel_Price    420212 non-null  float64
 6   MarkDown1     420212 non-null  float64
 7   MarkDown2     420212 non-null  float64
 8   MarkDown3     420212 non-null  float64
 9   MarkDown4     420212 non-null  float64
 10  MarkDown5     420212 non-null  float64
 11  CPI           420212 non-null  float64
 12  Unemployment  420212 non-null  float64
 13  Type          420212 non-null  int64  
 14  Size          420212 non-null  int64  
 15  Year          420212 non-null  int32  
 16  Month         420212 non-null  int32  
 17  Week          420212 non-null  UInt32 
 18  Day      

In [463]:
X = n_data.drop(columns=['Weekly_Sales'])
y = n_data['Weekly_Sales']

In [464]:
scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

# Convert back to DataFrame
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)

print(X_scaled.head())

      Store     Dept  IsHoliday  Temperature  Fuel_Price  MarkDown1  \
0 -1.657562 -1.41734  -0.275079    -0.963831   -1.720520  -0.427912   
1 -1.657562 -1.41734   3.635321    -1.169817   -1.772863  -0.427912   
2 -1.657562 -1.41734  -0.275079    -1.092844   -1.847014  -0.427912   
3 -1.657562 -1.41734  -0.275079    -0.729657   -1.744510  -0.427912   
4 -1.657562 -1.41734  -0.275079    -0.736704   -1.604931  -0.427912   

   MarkDown2  MarkDown3  MarkDown4  MarkDown5       CPI  Unemployment  \
0  -0.173118   -0.08472   -0.27811  -0.395322  1.018422      0.078331   
1  -0.173118   -0.08472   -0.27811  -0.395322  1.022146      0.078331   
2  -0.173118   -0.08472   -0.27811  -0.395322  1.023345      0.078331   
3  -0.173118   -0.08472   -0.27811  -0.395322  1.024124      0.078331   
4  -0.173118   -0.08472   -0.27811  -0.395322  1.024903      0.078331   

       Type      Size     Year     Month      Week       Day  
0 -0.884596  0.238802 -1.21528 -1.371979 -1.471715 -1.219483  
1 -0.884

In [465]:
# Calculating VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X_scaled.columns
vif_data["VIF"] = [variance_inflation_factor(X_scaled.values, i) for i in range(X_scaled.shape[1])]

vif_data

,Feature,VIF
0,Store,1.133421
1,Dept,1.000693
2,IsHoliday,1.268749
3,Temperature,1.390002
4,Fuel_Price,4.386579
5,MarkDown1,4.490032
6,MarkDown2,1.159667
7,MarkDown3,1.109055
8,MarkDown4,3.567887
9,MarkDown5,1.397258


In [466]:
X_scaled.drop(['Month','Week','Day'], axis=1, inplace = True)

In [467]:
# Calculating VIF for each feature
vif_data = pd.DataFrame()
vif_data["Feature"] = X_scaled.columns
vif_data["VIF"] = [variance_inflation_factor(X_scaled.values, i) for i in range(X_scaled.shape[1])]

vif_data

,Feature,VIF
0,Store,1.132080
1,Dept,1.000692
2,IsHoliday,1.156419
3,Temperature,1.177270
4,Fuel_Price,3.356005
5,MarkDown1,4.451076
6,MarkDown2,1.135106
7,MarkDown3,1.091150
8,MarkDown4,3.528496
9,MarkDown5,1.344554


## 6. Model Building & Machine Learning

In this section, we build a baseline machine learning model to forecast weekly sales. We will:


- Split the data into training and testing sets
- Train a model using RandomForestregressor
- Evaluate the model performance using R²

In [468]:
# Split Data into Train & Test Sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [469]:
# Train Model
rf = RandomForestRegressor(n_estimators=40, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Predict
y_train_pred = rf.predict(X_train)
y_test_pred = rf.predict(X_test)

# Evaluate
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"Random Forest - Train R²: {train_r2:.4f}, Test R²: {test_r2:.4f}")

Random Forest - Train R²: 0.8803, Test R²: 0.8715
